# Label-sensitivity experiment

The question: how much do the results change depending on how we treat the -999 rows in Dataset 2? Some work treats -999 as normal (for example a 2026 hybrid XGBoost / Isolation Forest / LSTM paper on BATADAL). But our audit (`docs/audit.md`) found about 273 real attack hours hidden in the -999 rows, including all of attacks 2 and 7.

Ridge never learns from labels, so the label rule can't change it. To test this, we need a model that learns from labels: a classifier trained on Dataset 2.

- Data used: `data/processed/dataset2_eval.csv` (Dataset 2 with `ATT_FLAG`, and our `label` and `attack_id` from the 7 attack windows).
- Dataset 1 and the Test set are not used.
- The full write-up is in `docs/label_experiment.md`. The notes here are short.

Steps:
1. Build the three label rules and the 7 blocks
2. Logistic regression, leave one attack out
3. Random forest, one seed
4. Random forest, 3 seeds
5. Save the results

## 1. The three label rules and the 7 blocks

Why: we want three versions of the training labels that only differ in what they do with the 273 hidden attack hours. We also need to split Dataset 2 into pieces, so that we can always test on an attack the model never trained on.

The three rules:
- **Rule 1, -999 as normal:** `ATT_FLAG = 1` is attack, everything else is normal. The 273 hidden attack hours become normal.
- **Rule 2, confirmed only:** `ATT_FLAG = 1` is attack, hours outside the 7 windows are normal, and the 273 hidden attack hours are left out (`NaN`). We use the Dataset 2 hours outside the windows as normal, not Dataset 1, because Dataset 1 is a different simulation year (2014 vs 2016).
- **Rule 3, rebuilt from the 7 windows:** every hour inside a window is attack (our `label` column). The 273 hidden hours become attack.

What the code does: loads `dataset2_eval.csv` and takes the 36 sensor columns (`L_`, `F_`, `P_`, `S_`) as `features`. It builds `y_rule1`, `y_rule2` and `y_rule3` and prints how many hours each rule has of each label. Then it cuts Dataset 2 into 7 blocks, one per attack. Each cut is halfway between the end of one attack and the start of the next. It prints the start, end, hours and attack hours of each block.

In [1]:
import numpy as np
import pandas as pd

d2 = pd.read_csv("../data/processed/dataset2_eval.csv", parse_dates=["DATETIME"])
features = [c for c in d2.columns if c.startswith(("L_", "F_", "P_", "S_"))]

d2["y_rule1"] = (d2["ATT_FLAG"] == 1).astype(int)

d2["y_rule2"] = np.nan
d2.loc[d2["ATT_FLAG"] == 1, "y_rule2"] = 1
d2.loc[d2["label"] == 0, "y_rule2"] = 0

d2["y_rule3"] = d2["label"]

for rule in ["y_rule1", "y_rule2", "y_rule3"]:
    print(rule, d2[rule].value_counts(dropna=False).to_dict())

pos = pd.Series(d2.index, index=d2.index)
in_attack = d2["attack_id"] > 0
first = pos[in_attack].groupby(d2["attack_id"]).min()
last = pos[in_attack].groupby(d2["attack_id"]).max()

cuts = [0]
for a in range(1, 7):
    cuts.append((last[a] + first[a + 1]) // 2 + 1)
cuts.append(len(d2))

d2["block"] = 0
for b in range(7):
    d2.loc[cuts[b]:cuts[b + 1] - 1, "block"] = b + 1

summary = d2.groupby("block").agg(start=("DATETIME", "min"), end=("DATETIME", "max"),
                                  hours=("DATETIME", "size"), attack_hours=("label", "sum"),
                                  attack=("attack_id", "max"))
print(len(features))
print(summary)

y_rule1 {0: 3958, 1: 219}
y_rule2 {0.0: 3685, nan: 273, 1.0: 219}
y_rule3 {0: 3685, 1: 492}
36
                    start                 end  hours  attack_hours  attack
block                                                                     
1     2016-07-04 00:00:00 2016-09-21 05:00:00   1902            50       1
2     2016-09-21 06:00:00 2016-10-03 09:00:00    292            24       2
3     2016-10-03 10:00:00 2016-10-20 19:00:00    418            60       3
4     2016-10-20 20:00:00 2016-11-14 16:00:00    597            94       4
5     2016-11-14 17:00:00 2016-12-02 17:00:00    433            60       5
6     2016-12-02 18:00:00 2016-12-12 09:00:00    232            94       6
7     2016-12-12 10:00:00 2016-12-25 00:00:00    303           110       7


**What this shows:**
- Rule 1: 219 attack and 3,958 normal hours.
- Rule 2: 219 attack and 3,685 normal hours. 273 hours are left out (`nan`).
- Rule 3: 492 attack and 3,685 normal hours.
- So the rules agree on everything except the 273 hidden hours: rule 1 calls them normal, rule 2 leaves them out, and rule 3 calls them attack.
- 36 features.

| Block | Start | End | Hours | Attack hours | Attack |
|---|---|---|---|---|---|
| 1 | 2016-07-04 00:00 | 2016-09-21 05:00 | 1,902 | 50 | 1 |
| 2 | 2016-09-21 06:00 | 2016-10-03 09:00 | 292 | 24 | 2 |
| 3 | 2016-10-03 10:00 | 2016-10-20 19:00 | 418 | 60 | 3 |
| 4 | 2016-10-20 20:00 | 2016-11-14 16:00 | 597 | 94 | 4 |
| 5 | 2016-11-14 17:00 | 2016-12-02 17:00 | 433 | 60 | 5 |
| 6 | 2016-12-02 18:00 | 2016-12-12 09:00 | 232 | 94 | 6 |
| 7 | 2016-12-12 10:00 | 2016-12-25 00:00 | 303 | 110 | 7 |

Caution: the blocks are very uneven. Block 1 is 1,902 hours with only 50 attack hours, because Dataset 2 starts more than 2 months before attack 1. Block 6 is 232 hours with 94 attack hours.

## 2. Logistic regression, leave one attack out

Why: we start with the simplest classifier. Leave one attack out means the model is always tested on an attack it never trained on, which is closer to real use.

What the code does: for each rule, runs 7 rounds. In each round, it tests on one block and trains on the other 6. The 24 hours on each side of the test block are left out of training (`gap = 24`), so the model can't learn from hours right next to the test block. Hours with no label (rule 2's hidden hours) are also left out of training. The model is `LogisticRegression(class_weight="balanced")`, so the rare attack hours count as much as the normal ones.

The test block is always graded with the rule 3 labels (`label`). So only the training labels change between the rules, and every rule is graded with the same answer key. The ROC-AUC and PR-AUC are computed once over all 4,177 hours, not per block, because the blocks are so uneven. Last, it prints the share of hours with a score above 0.5 for each attack (`attack_id` 0 means outside the attacks).

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

gap = 24
rules = ["y_rule1", "y_rule2", "y_rule3"]

for rule in rules:
    d2[f"score_{rule}"] = np.nan
    for b in range(1, 8):
        test = d2["block"] == b
        start, end = d2.index[test].min(), d2.index[test].max()
        near = (d2.index >= start - gap) & (d2.index <= end + gap)
        train = ~near & d2[rule].notna()

        model = LogisticRegression(class_weight="balanced", max_iter=1000)
        model.fit(d2.loc[train, features], d2.loc[train, rule])
        d2.loc[test, f"score_{rule}"] = model.predict_proba(d2.loc[test, features])[:, 1]

rows = {}
for rule in rules:
    s = d2[f"score_{rule}"]
    rows[rule] = {"ROC-AUC": roc_auc_score(d2["label"], s),
                  "PR-AUC": average_precision_score(d2["label"], s)}
print(pd.DataFrame(rows).round(3))

per_attack = {}
for rule in rules:
    per_attack[rule] = (d2[f"score_{rule}"] > 0.5).groupby(d2["attack_id"]).mean()
print(pd.DataFrame(per_attack).round(2))

         y_rule1  y_rule2  y_rule3
ROC-AUC    0.524    0.535    0.599
PR-AUC     0.247    0.255    0.336
           y_rule1  y_rule2  y_rule3
attack_id                           
0             0.11     0.10     0.15
1             0.12     0.12     0.70
2             0.42     0.42     0.38
3             0.40     0.40     0.43
4             0.48     0.47     0.44
5             0.40     0.40     0.57
6             0.14     0.14     0.27
7             0.11     0.12     0.20


**What this shows:**

| | Rule 1 | Rule 2 | Rule 3 |
|---|---|---|---|
| ROC-AUC | 0.524 | 0.535 | 0.599 |
| PR-AUC | 0.247 | 0.255 | 0.336 |

- Logistic regression is weak overall. A ROC-AUC of 0.524 is close to guessing.
- But the order is the same for both measures: rule 1 < rule 2 < rule 3.
- At the 0.5 cutoff, 10% to 15% of normal hours are flagged (0.11, 0.10 and 0.15 for `attack_id` 0).
- The biggest change is attack 1: 0.12 under rules 1 and 2, and 0.70 under rule 3.

Caution: with 10% to 15% of normal hours flagged, the per-attack shares are not a good measure of detection. They only show the pattern. Some attacks even go down a little under rule 3 (attack 2 from 0.42 to 0.38, attack 4 from 0.48 to 0.44).

## 3. Random forest, one seed

Why: logistic regression can only draw a straight line between attack and normal. A random forest can learn more complex patterns, so it is a fairer test of what labels can teach.

What the code does: the same 7 rounds, 24-hour gap and grading as step 2, but with `RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced", random_state=0)`. It prints the ROC-AUC and PR-AUC for each rule, and the share of hours above 0.5 for each attack. The scores are saved as `rf_y_rule1` to `rf_y_rule3`.

In [3]:
from sklearn.ensemble import RandomForestClassifier

for rule in rules:
    col = f"rf_{rule}"
    d2[col] = np.nan
    for b in range(1, 8):
        test = d2["block"] == b
        start, end = d2.index[test].min(), d2.index[test].max()
        near = (d2.index >= start - gap) & (d2.index <= end + gap)
        train = ~near & d2[rule].notna()

        model = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                       class_weight="balanced", random_state=0, n_jobs=-1)
        model.fit(d2.loc[train, features], d2.loc[train, rule])
        d2.loc[test, col] = model.predict_proba(d2.loc[test, features])[:, 1]

rows = {}
for rule in rules:
    s = d2[f"rf_{rule}"]
    rows[rule] = {"ROC-AUC": roc_auc_score(d2["label"], s),
                  "PR-AUC": average_precision_score(d2["label"], s)}
print(pd.DataFrame(rows).round(3))

per_attack = {}
for rule in rules:
    per_attack[rule] = (d2[f"rf_{rule}"] > 0.5).groupby(d2["attack_id"]).mean()
print(pd.DataFrame(per_attack).round(2))

         y_rule1  y_rule2  y_rule3
ROC-AUC    0.700    0.704    0.757
PR-AUC     0.274    0.274    0.458
           y_rule1  y_rule2  y_rule3
attack_id                           
0             0.01     0.01     0.01
1             0.00     0.00     0.38
2             0.00     0.00     0.12
3             0.02     0.02     0.10
4             0.04     0.05     0.10
5             0.25     0.25     0.27
6             0.01     0.00     0.70
7             0.02     0.02     0.03


**What this shows:**

| | Rule 1 | Rule 2 | Rule 3 |
|---|---|---|---|
| ROC-AUC | 0.700 | 0.704 | 0.757 |
| PR-AUC | 0.274 | 0.274 | 0.458 |

- The random forest is much stronger than logistic regression under every rule.
- Only about 1% of normal hours are above 0.5 (0.01 for `attack_id` 0 under all three rules). So the per-attack shares mean more here than in step 2.
- Attack 1 goes from 0.00 (rules 1 and 2) to 0.38 (rule 3). Attack 6 goes from 0.01 and 0.00 to 0.70.
- Rules 1 and 2 are almost the same.

Caution: this is one seed. Step 4 checks if the result depends on it.

## 4. Random forest with 3 seeds

Why: a random forest is random, so one seed could be lucky. We repeat step 3 with seeds 0, 1 and 2.

What the code does: the same as step 3, for each seed and each rule. It saves the ROC-AUC and PR-AUC of each run in `aucs`, and the share of hours above 0.5 for each attack in `shares`. It prints the mean and standard deviation of the AUCs for each rule, the mean share for each attack, and the smallest and biggest share over the 3 seeds.

In [4]:
seeds = [0, 1, 2]
auc_rows = []
attack_rows = []

for seed in seeds:
    for rule in rules:
        scores = pd.Series(np.nan, index=d2.index)
        for b in range(1, 8):
            test = d2["block"] == b
            start, end = d2.index[test].min(), d2.index[test].max()
            near = (d2.index >= start - gap) & (d2.index <= end + gap)
            train = ~near & d2[rule].notna()

            model = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                           class_weight="balanced", random_state=seed, n_jobs=-1)
            model.fit(d2.loc[train, features], d2.loc[train, rule])
            scores[test] = model.predict_proba(d2.loc[test, features])[:, 1]

        auc_rows.append({"seed": seed, "rule": rule,
                         "ROC-AUC": roc_auc_score(d2["label"], scores),
                         "PR-AUC": average_precision_score(d2["label"], scores)})
        share = (scores > 0.5).groupby(d2["attack_id"]).mean()
        for a, v in share.items():
            attack_rows.append({"seed": seed, "rule": rule, "attack": a, "share": v})

aucs = pd.DataFrame(auc_rows)
print(aucs.groupby("rule")[["ROC-AUC", "PR-AUC"]].agg(["mean", "std"]).round(3))

shares = pd.DataFrame(attack_rows)
print(shares.pivot_table(index="attack", columns="rule", values="share", aggfunc="mean").round(2))
print(shares.pivot_table(index="attack", columns="rule", values="share", aggfunc=["min", "max"]).round(2))

        ROC-AUC        PR-AUC       
           mean    std   mean    std
rule                                
y_rule1   0.696  0.004  0.269  0.005
y_rule2   0.705  0.001  0.272  0.002
y_rule3   0.756  0.003  0.452  0.010
rule    y_rule1  y_rule2  y_rule3
attack                           
0          0.01     0.01     0.01
1          0.00     0.00     0.40
2          0.00     0.00     0.12
3          0.02     0.02     0.11
4          0.04     0.05     0.06
5          0.25     0.26     0.27
6          0.00     0.00     0.69
7          0.02     0.02     0.03
           min                     max                
rule   y_rule1 y_rule2 y_rule3 y_rule1 y_rule2 y_rule3
attack                                                
0         0.01    0.01    0.01    0.01    0.01    0.01
1         0.00    0.00    0.38    0.00    0.00    0.42
2         0.00    0.00    0.08    0.00    0.00    0.17
3         0.02    0.02    0.10    0.02    0.02    0.12
4         0.03    0.04    0.04    0.05    0.05    0.1

**What this shows:**

| | Rule 1 | Rule 2 | Rule 3 |
|---|---|---|---|
| ROC-AUC (mean) | 0.696 | 0.705 | 0.756 |
| ROC-AUC (std) | 0.004 | 0.001 | 0.003 |
| PR-AUC (mean) | 0.269 | 0.272 | 0.452 |
| PR-AUC (std) | 0.005 | 0.002 | 0.010 |

Share of hours above 0.5, smallest to biggest over the 3 seeds:

| Attack | Rule 1 | Rule 2 | Rule 3 |
|---|---|---|---|
| 0 (outside attacks) | 0.01 | 0.01 | 0.01 |
| 1 | 0.00 | 0.00 | 0.38 to 0.42 |
| 2 | 0.00 | 0.00 | 0.08 to 0.17 |
| 3 | 0.02 | 0.02 | 0.10 to 0.12 |
| 4 | 0.03 to 0.05 | 0.04 to 0.05 | 0.04 to 0.10 |
| 5 | 0.25 | 0.25 to 0.27 | 0.27 to 0.28 |
| 6 | 0.00 to 0.01 | 0.00 | 0.68 to 0.70 |
| 7 | 0.02 | 0.02 | 0.02 to 0.04 |

- The std is 0.01 or less, so the result doesn't depend on the seed.
- **The label rule matters most when the unlabeled hours are the only examples of a kind of attack.**
  - Attacks 1 and 2 are similar (both are about tank 7), and attack 2 has no labels. When we test on attack 1, the only other tank 7 attack is attack 2. Under rules 1 and 2 it isn't an attack example, so attack 1 gets 0.00. Under rule 3 it gets 0.38 to 0.42.
  - Attacks 6 and 7 are similar (both slow down pump 7), and attack 7 has no labels. Attack 6 goes from 0.00 to 0.01 under rules 1 and 2, to 0.68 to 0.70 under rule 3. (Attack 5 also slows pump 7, but most of its hours are unlabeled too, see `docs/audit.md`.)
  - Attacks 3, 4 and 5 each have a similar attack that is at least partly labeled under every rule (attack 3 has attack 4, attack 4 has attack 3, and attack 5 has attack 6), so the classifier gets some examples of that kind of attack either way. They change much less: attack 3 goes from 0.02 to about 0.11 and attack 4 from about 0.04 to 0.06, both still low, and attack 5 stays about 0.25 to 0.27.
- **Rules 1 and 2 give almost the same results** (ROC-AUC 0.696 and 0.705, PR-AUC 0.269 and 0.272). So the loss comes from the hidden hours missing as attack examples, not from calling them normal.
- **PR-AUC shows the difference much more than ROC-AUC:** 0.269 to 0.452, compared with 0.696 to 0.756. PR-AUC looks at the top of the ranking, and attacks are rare, so it reacts more when the model learns to put attack hours first.
- **Even the best classifier is below Ridge.** Rule 3 gives ROC-AUC 0.756. Ridge gets 0.864 (`docs/ridge.md`, section 11) and never saw a label. With only 7 attacks, learning normal behavior works better than learning from labeled attacks.
- **Attacks 3 and 7 stay hard** under every rule and seed (at most 0.12 and 0.04). Ridge also only catches about 17% and 24% of their hours (10 of 60 and 26 of 110, `docs/ridge.md`, section 7).

Caution:
- These are simple classifiers, and there are only 7 attacks, so each round depends on a few attack examples.
- The 0.5 cutoff is not a calibrated threshold. The per-attack shares only compare the rules with each other. They are not detection results.
- Rules 2 and 3 assume every hour outside the 7 windows is normal. This depends on the list of attacks being complete.
- Ridge's ROC-AUC comes from a different setup (trained on Dataset 1, no leave one attack out), so the comparison is rough.

## 5. Save the results

Why: so the scores and tables can be checked later without running everything again.

What the code does: saves three files in `../models/Label_Experiment/`. It renames the columns to clearer names (for example `rf_y_rule1` becomes `rf_seed0_score_rule1`), then reads the scores file back and prints its shape, the summary table, the shape of the per-attack table and the folder contents.

In [5]:
import os

out = "../models/Label_Experiment"
os.makedirs(out, exist_ok=True)

names = {}
for r in [1, 2, 3]:
    names[f"y_rule{r}"] = f"train_label_rule{r}"
    names[f"score_y_rule{r}"] = f"lr_score_rule{r}"
    names[f"rf_y_rule{r}"] = f"rf_seed0_score_rule{r}"
keep = ["DATETIME", "ATT_FLAG", "label", "attack_id", "block"] + list(names)
d2[keep].rename(columns=names).to_csv(f"{out}/label_experiment_scores.csv", index=False)

lr_rows = []
for rule in rules:
    s = d2[f"score_{rule}"]
    lr_rows.append({"model": "logistic_regression", "rule": rule, "seeds": 1,
                    "roc_auc_mean": roc_auc_score(d2["label"], s), "roc_auc_std": np.nan,
                    "pr_auc_mean": average_precision_score(d2["label"], s), "pr_auc_std": np.nan})
rf = aucs.groupby("rule").agg(roc_auc_mean=("ROC-AUC", "mean"), roc_auc_std=("ROC-AUC", "std"),
                              pr_auc_mean=("PR-AUC", "mean"), pr_auc_std=("PR-AUC", "std")).reset_index()
rf["model"] = "random_forest"
rf["seeds"] = 3
summary = pd.concat([pd.DataFrame(lr_rows), rf], ignore_index=True)
summary.to_csv(f"{out}/label_experiment_summary.csv", index=False)

lr_attack = []
for rule in rules:
    sh = (d2[f"score_{rule}"] > 0.5).groupby(d2["attack_id"]).mean()
    for a, v in sh.items():
        lr_attack.append({"model": "logistic_regression", "rule": rule, "attack": a,
                          "mean": v, "min": v, "max": v})
rf_attack = shares.groupby(["rule", "attack"])["share"].agg(["mean", "min", "max"]).reset_index()
rf_attack["model"] = "random_forest"
per_attack = pd.concat([pd.DataFrame(lr_attack), rf_attack], ignore_index=True)
per_attack.to_csv(f"{out}/label_experiment_per_attack.csv", index=False)

print(pd.read_csv(f"{out}/label_experiment_scores.csv").shape)
print(summary.round(3))
print(per_attack.shape)
print(os.listdir(out))

(4177, 14)
                 model     rule  seeds  roc_auc_mean  roc_auc_std  \
0  logistic_regression  y_rule1      1         0.524          NaN   
1  logistic_regression  y_rule2      1         0.535          NaN   
2  logistic_regression  y_rule3      1         0.599          NaN   
3        random_forest  y_rule1      3         0.696        0.004   
4        random_forest  y_rule2      3         0.705        0.001   
5        random_forest  y_rule3      3         0.756        0.003   

   pr_auc_mean  pr_auc_std  
0        0.247         NaN  
1        0.255         NaN  
2        0.336         NaN  
3        0.269       0.005  
4        0.272       0.002  
5        0.452       0.010  
(48, 6)
['label_experiment_per_attack.csv', 'label_experiment_scores.csv', 'label_experiment_summary.csv']


**What this shows:**
- The scores file has 4,177 rows and 14 columns. The per-attack table has 48 rows (2 models × 3 rules × 8 rows: attacks 1 to 7 and 0 for outside the attacks).
- The summary table matches steps 2 and 4.
- The three files in `models/Label_Experiment/`:

| File | What it holds |
|---|---|
| `label_experiment_scores.csv` | One row per Dataset 2 hour: `DATETIME`, `ATT_FLAG`, `label`, `attack_id`, `block`, the training labels of each rule (`train_label_rule1` to `3`), the logistic regression scores (`lr_score_rule1` to `3`) and the random forest scores with seed 0 (`rf_seed0_score_rule1` to `3`) |
| `label_experiment_summary.csv` | ROC-AUC and PR-AUC for each model and rule. Mean and std over 3 seeds for the random forest. The logistic regression has 1 run, so its std is empty |
| `label_experiment_per_attack.csv` | Share of hours above 0.5 for each model, rule and attack, with the mean, smallest and biggest over the seeds. For logistic regression all three are the same |

## End of notebook

The full write-up is in `docs/label_experiment.md`.